# 🤖 vla-hands — Quick Start (Colab / Kaggle)

This notebook gives a VLM **hands** by grafting lightweight action heads onto its
frozen hidden states — turning it into a VLA (Vision-Language-Action) model.

We'll train four appendage types on tiny environments:
| Appendage | Environment | Task |
|-----------|-------------|------|
| 🕹️ Joystick | Target Nav / Spaceship | Continuous 2D navigation |
| 🎮 D-pad | Grid World / Maze | Discrete directional movement |
| 🔘 Button | Color Press | Press when circle matches target color |
| 🎛️ MultiButton | MCQ | Answer multiple-choice visual questions |

**Runtime:** T4 GPU on Colab Free / P100 on Kaggle  
**Model:** `HuggingFaceTB/SmolVLM-256M-Instruct` (~1 GB, fits on T4 easily)  
**Training time:** ~10–15 min for a quick demo run

## 1 · Setup

In [ ]:
# Install the package directly from GitHub
# (change the URL/branch if you have a fork)
!pip install -q git+https://github.com/jerod92/project-h.git@claude/vla-robotic-hands-platform-kGiza

# Or, if you've cloned it locally (e.g. on Kaggle with a dataset):
# import sys; sys.path.insert(0, '/kaggle/input/vla-hands/project-h')

print('✅ vla-hands installed')

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from PIL import Image

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2 · Visualise the Environments

Before training anything, let's see what each environment looks like and verify
the expert policy works as expected.

In [ ]:
from vla_hands import (
    TargetNavEnvironment,
    SpaceshipNavEnvironment,
    GridWorldEnvironment,
    MazeEnvironment,
    ButtonPressEnvironment,
    MCQButtonEnvironment,
    PointingEnvironment,
)

envs = {
    'Target Nav (joystick)':  TargetNavEnvironment(width=200, height=200),
    'Spaceship (joystick)':   SpaceshipNavEnvironment(width=200, height=200),
    'Grid World (d-pad)':     GridWorldEnvironment(grid_size=6),
    'Maze (d-pad)':           MazeEnvironment(rows=6, cols=6),
    'Color Press (button)':   ButtonPressEnvironment(width=200, height=200),
    'MCQ (multi-button)':     MCQButtonEnvironment(width=240, height=200),
    'Pointing (touchscreen)': PointingEnvironment(width=200, height=200),
}

# Dynamic prompts: different wording every reset()
print('Dynamic prompt examples for PointingEnvironment:')
env_pt = PointingEnvironment()
for i in range(3):
    obs = env_pt.reset(seed=i)
    print(f'  seed={i}: {env_pt.prompt}')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (name, env) in zip(axes.flat, envs.items()):
    obs = env.reset(seed=42)
    ax.imshow(obs)
    ax.set_title(name, fontsize=10)
    ax.axis('off')
axes.flat[-1].set_visible(False)   # 7 envs, 8 slots
plt.suptitle('VLA-Hands: Available Training Environments', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Roll out the expert policy for a few steps and render the trajectory
def render_expert_rollout(env, n_steps=12, cols=6, seed=7):
    obs = env.reset(seed=seed)
    frames = [obs]
    for _ in range(n_steps - 1):
        action = env.expert_action()
        result = env.step(action)
        frames.append(result.observation)
        if result.done:
            break

    rows = (len(frames) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
    axes = np.array(axes).flat
    for ax, frame in zip(axes, frames):
        ax.imshow(frame)
        ax.axis('off')
    for ax in list(axes)[len(frames):]:
        ax.set_visible(False)
    plt.suptitle(f'{type(env).__name__} — Expert Rollout')
    plt.tight_layout()
    plt.show()

render_expert_rollout(TargetNavEnvironment(224, 224), n_steps=8)

In [ ]:
render_expert_rollout(MazeEnvironment(rows=5, cols=5), n_steps=20, cols=5)

In [ ]:
# Expert baseline — the theoretical ceiling for each environment
from vla_hands import run_expert_baseline

print('Expert baselines (20 episodes each):')
for name, env in [
    ('TargetNav', TargetNavEnvironment()),
    ('Spaceship', SpaceshipNavEnvironment()),
    ('GridWorld', GridWorldEnvironment()),
    ('Maze 7x7', MazeEnvironment()),
    ('ColorPress', ButtonPressEnvironment()),
    ('MCQ', MCQButtonEnvironment()),
]:
    r = run_expert_baseline(env, n_episodes=20)
    print(f'  {name:12s} success={r.success_rate:.0%}  reward={r.mean_reward:+.1f}')

## 3 · Load the VLM

We use **SmolVLM-256M-Instruct** — a tiny but capable vision-language model  
that fits in ~2 GB of VRAM and loads in under a minute.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'
# Alternatives (larger, better visual understanding):
# MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'  # ~2.5 GB
# MODEL_ID = 'HuggingFaceTB/SmolVLM-Instruct'       # ~4 GB (2B params)

print(f'Loading {MODEL_ID}...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,   # use torch.float16 if VRAM is tight
)

n_params = sum(p.numel() for p in vlm.parameters())

# SmolVLM (and other multimodal models) store language hidden size in text_config.
# Fall back to top-level hidden_size for simpler single-config models.
text_cfg = getattr(vlm.config, 'text_config', None)
hidden_dim = (
    text_cfg.hidden_size
    if text_cfg is not None and hasattr(text_cfg, 'hidden_size')
    else vlm.config.hidden_size
)

print(f'Parameters:  {n_params/1e6:.0f}M')
print(f'Hidden dim:  {hidden_dim}')
print(f'Model type:  {type(vlm).__name__}')

## 3b · Optional: Apply LoRA for Faster Training

LoRA (Low-Rank Adaptation) adds tiny trainable rank-decomposition matrices to the
VLM's attention layers, reducing trainable parameters by **~100×** versus full fine-tuning.

| Mode | Trainable params | Memory | Speed |
|------|-----------------|--------|-------|
| Frozen backbone | ~0 (appendage only) | low | fastest |
| **LoRA** | **~0.5% of VLM** | **low** | **fast** |
| Last-2 layers | ~2% of VLM | medium | medium |
| Last-6 layers | ~6% of VLM | higher | slower |

**When to use LoRA:** when you want the VLM representation to adapt to the task
but can't afford to unfreeze full layers. Best combined with `DEFAULT_CURRICULUM`.

In [ ]:
# ── LoRA (optional — skip this cell for the frozen-backbone demo) ─────────────
# Requires:  pip install peft>=0.10.0
#
# If you want the VLM backbone to adapt during training, run this cell.
# The `vlm` variable will be replaced with a LoRA-wrapped version.
# Downstream grafts (VLAGraft) remain identical — no interface changes.

USE_LORA = False   # ← set True to enable LoRA

if USE_LORA:
    try:
        from vla_hands import apply_lora, LoRAConfig, lora_parameter_count

        lora_cfg = LoRAConfig(
            r=8,        # rank — higher = more expressive, higher VRAM
            alpha=16,   # scaling (alpha/r = effective LR multiplier)
            dropout=0.05,
            target_modules='auto',  # auto-detects q_proj, v_proj for SmolVLM
        )

        vlm = apply_lora(vlm, lora_cfg)   # wraps vlm in peft.PeftModel

        counts = lora_parameter_count(vlm)
        print(f'LoRA applied!')
        print(f'  Trainable:  {counts["trainable"]:,} params')
        print(f'  Total:      {counts["total"]:,} params')
        print(f'  % trainable: {counts["pct_trainable"]:.2f}%')

    except ImportError:
        print('peft not installed. Run: pip install peft>=0.10.0')
else:
    print('LoRA skipped (USE_LORA=False). VLM backbone fully frozen.')
    print('Set USE_LORA=True to enable parameter-efficient fine-tuning.')

## 4 · Create Grafts

Each graft pairs a **VLM** with an **appendage** (action head).  
The VLM backbone is frozen to start — we only train the tiny MLP.

In [ ]:
from vla_hands import (
    VLAGraft, GraftConfig,
    JoystickAppendage,
    DPadAppendage,
    ButtonAppendage,
    MultiButtonAppendage,
)

# Shared config: extract features from last token position
cfg = GraftConfig(feature_extraction='last')

# Four grafts sharing the same frozen VLM backbone
joystick_graft = VLAGraft(vlm=vlm, appendage=JoystickAppendage(hidden_dim), config=cfg)
dpad_graft     = VLAGraft(vlm=vlm, appendage=DPadAppendage(hidden_dim),     config=cfg)
button_graft   = VLAGraft(vlm=vlm, appendage=ButtonAppendage(hidden_dim),   config=cfg)
mcq_graft      = VLAGraft(vlm=vlm, appendage=MultiButtonAppendage(hidden_dim, n_buttons=4,
                                              labels=['A','B','C','D']), config=cfg)

print(joystick_graft)
print(f'\nAppendage sizes:')
for name, g in [('joystick', joystick_graft), ('dpad', dpad_graft),
                ('button', button_graft), ('mcq_4btn', mcq_graft)]:
    n = g.appendage.num_parameters()
    print(f'  {name:12s} {n:,} params  ({n/1e3:.0f}K)')

## 4b · Touchscreen Graft (with Vision Skip Connection)

The **TouchscreenAppendage** outputs absolute `(x, y)` coordinates ∈ [0, 1]² —
where to *tap* on the image — rather than a directional joystick value.

It optionally receives a **skip connection** from the VLM's vision encoder:
raw patch embeddings are mean-pooled and concatenated with the LLM hidden state
before the prediction head.  This gives the head direct access to spatial image
features that might get compressed away by the LLM decoder.

```
image → [Vision Encoder] ──────────────────────────── skip ─────┐
                       → [LLM Decoder] → last token hidden state ┤ → MLP → (x, y)
                                                                  ┘
```

In [ ]:
from vla_hands import TouchscreenAppendage, VLAGraft, GraftConfig

# Detect the vision encoder's hidden size for the skip connection
vision_dim = VLAGraft.detect_vision_dim(vlm)
print(f'Vision encoder hidden dim: {vision_dim}')
# SmolVLM-256M uses SigLIP vision encoder → 1152

# Touchscreen WITHOUT skip connection (LLM features only)
ts_appendage_plain = TouchscreenAppendage(
    hidden_dim=hidden_dim,
    vision_dim=None,      # no skip connection
)

# Touchscreen WITH vision skip connection (recommended)
ts_appendage = TouchscreenAppendage(
    hidden_dim=hidden_dim,
    vision_dim=vision_dim,   # skip from vision encoder
    intermediate_dim=256,
)

print(ts_appendage)
print(f'\nParameters:')
print(f'  Plain (LLM only):    {ts_appendage_plain.num_parameters():,}')
print(f'  With vision skip:    {ts_appendage.num_parameters():,}')
print(f'  needs_vision_features: {ts_appendage.needs_vision_features}')

# VLAGraft automatically detects needs_vision_features=True and registers
# a forward hook on the vision encoder to capture patch embeddings.
ts_graft = VLAGraft(
    vlm=vlm,
    appendage=ts_appendage,
    config=GraftConfig(feature_extraction='last'),
)
print(f'\n{ts_graft}')

# The vision hook status is shown indirectly — if the hook was registered,
# the graft will pass vision_features to the appendage during forward passes.
has_hook = ts_graft._vision_hook is not None
print(f'Vision hook registered: {has_hook}')

## 5 · Train

We run a BC (Behavioral Cloning) warm-start followed by brief RL fine-tuning.

Training uses the **`QUICK_CURRICULUM`** (appendage weights only, VLM frozen)
so it's fast enough to run in a free Colab session.

To get serious performance: switch to `DEFAULT_CURRICULUM` and increase steps.

In [ ]:
from vla_hands import TrainingCurriculum, CurriculumConfig
from vla_hands import QUICK_CURRICULUM

def quick_train(graft, env, bc_steps=200, rl_steps=50, tag=''):
    """Train one graft+environment pair and return metrics."""
    config = CurriculumConfig(
        bc_steps=bc_steps,
        rl_steps=rl_steps,
        appendage_lr=3e-4,     # higher LR for fast convergence on tiny appendage MLPs
        device=device,
        save_dir=f'model_checkpoints/{tag or type(env).__name__}',
        freezing_stages=QUICK_CURRICULUM,   # keep VLM fully frozen for speed
        eval_every=bc_steps // 4,
        log_every=bc_steps // 10,
        eval_episodes=5,
    )
    curriculum = TrainingCurriculum(graft, processor, env, config)
    return curriculum.run()

print('Training helpers ready.')

In [ ]:
# ── Joystick: Target Navigation ──────────────────────────────────────────────
print('='*60)
print('Training: Joystick → Target Navigation')
print('='*60)

target_env = TargetNavEnvironment(width=224, height=224, max_steps=80)
joystick_metrics = quick_train(joystick_graft, target_env, bc_steps=300, rl_steps=100,
                               tag='joystick_target')

In [ ]:
# ── D-pad: Grid World ────────────────────────────────────────────────────────
print('='*60)
print('Training: D-pad → Grid World')
print('='*60)

grid_env = GridWorldEnvironment(grid_size=6)
dpad_metrics = quick_train(dpad_graft, grid_env, bc_steps=300, rl_steps=100,
                           tag='dpad_grid')

In [ ]:
# ── Button: Color Recognition ────────────────────────────────────────────────
print('='*60)
print('Training: Button → Color Press')
print('='*60)

color_env = ButtonPressEnvironment(difficulty='easy')
button_metrics = quick_train(button_graft, color_env, bc_steps=200, rl_steps=0,
                             tag='button_color')

In [ ]:
# ── MultiButton: MCQ ─────────────────────────────────────────────────────────
print('='*60)
print('Training: MultiButton → Multiple Choice Questions')
print('='*60)

mcq_env = MCQButtonEnvironment(question_type='dots')
mcq_metrics = quick_train(mcq_graft, mcq_env, bc_steps=200, rl_steps=0,
                          tag='mcq_dots')

## 6 · Benchmark & Visualise Results

In [ ]:
def plot_metrics(metrics_dict, key='bc/loss', title='Training Loss'):
    fig, axes = plt.subplots(1, len(metrics_dict), figsize=(5*len(metrics_dict), 4))
    if len(metrics_dict) == 1:
        axes = [axes]
    for ax, (name, metrics) in zip(axes, metrics_dict.items()):
        bc_metrics = metrics.get('bc', metrics)
        steps = [m['step'] for m in bc_metrics if key in m]
        vals  = [m[key]  for m in bc_metrics if key in m]
        ax.plot(steps, vals, linewidth=2)
        ax.set_title(name)
        ax.set_xlabel('Step')
        ax.set_ylabel(key)
        ax.grid(alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_metrics({
    'Joystick': joystick_metrics,
    'D-pad':    dpad_metrics,
    'Button':   button_metrics,
    'MCQ':      mcq_metrics,
}, key='bc/loss', title='BC Training Loss')

In [ ]:
from vla_hands import BenchmarkSuite

results = {}
benchmark_pairs = [
    ('Joystick',  joystick_graft, TargetNavEnvironment()),
    ('D-pad',     dpad_graft,     GridWorldEnvironment(grid_size=6)),
    ('Button',    button_graft,   ButtonPressEnvironment()),
    ('MCQ',       mcq_graft,      MCQButtonEnvironment()),
]

for name, graft, env in benchmark_pairs:
    suite = BenchmarkSuite(graft, processor, [env], device=device)
    r = suite.run_benchmark(env, n_episodes=15)
    results[name] = r
    print(f'{name:12s}  success={r.success_rate:.0%}  reward={r.mean_reward:+.1f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

names    = list(results.keys())
sr_vals  = [results[n].success_rate * 100 for n in names]
rw_vals  = [results[n].mean_reward for n in names]

bars = ax1.bar(names, sr_vals, color=['#4C9BE8','#5BC46A','#E8804C','#C45BE8'])
ax1.set_ylabel('Success Rate (%)')
ax1.set_title('Success Rate per Appendage')
ax1.set_ylim(0, 110)
for bar, val in zip(bars, sr_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}%', ha='center', va='bottom', fontsize=11)

ax2.bar(names, rw_vals, color=['#4C9BE8','#5BC46A','#E8804C','#C45BE8'])
ax2.set_ylabel('Mean Episode Reward')
ax2.set_title('Mean Reward per Appendage')
ax2.axhline(0, color='gray', linewidth=0.5)

plt.suptitle('vla-hands Benchmark Results', fontsize=13)
plt.tight_layout()
plt.show()

## 7 · Interactive Inference

Run the trained grafts on fresh episodes and see what they predict.

In [ ]:
def run_inference_demo(graft, env, n_steps=8, seed=99):
    """Run a few steps of inference and display side-by-side with predicted action."""
    from vla_hands.training.trainer import _preprocess

    graft.eval()
    obs = env.reset(seed=seed)
    frames, actions = [obs], []

    for step in range(n_steps):
        with torch.no_grad():
            inputs = _preprocess(processor, obs, env.prompt, device)
            out = graft(**inputs)
            raw_action = out['action']

        decoded = graft.appendage.decode(raw_action)
        actions.append(decoded)

        # Get concrete action for env.step
        if hasattr(graft.appendage, 'argmax'):
            action_val = int(graft.appendage.argmax(raw_action).item())
        else:
            action_val = raw_action.squeeze(0).cpu().tolist()

        result = env.step(action_val)
        obs = result.observation
        frames.append(obs)
        if result.done:
            print(f'  Episode ended at step {step+1} — success={result.info.get("success")}')
            break

    # Display
    fig, axes = plt.subplots(1, len(frames), figsize=(3*len(frames), 3))
    for i, (ax, frame) in enumerate(zip(axes, frames)):
        ax.imshow(frame)
        ax.axis('off')
        if i < len(actions):
            ax.set_title(str(actions[i])[:18], fontsize=7)
    plt.suptitle(f'{type(graft.appendage).__name__} on {type(env).__name__}')
    plt.tight_layout()
    plt.show()


print('Joystick inference:')
run_inference_demo(joystick_graft, TargetNavEnvironment(), n_steps=8)

print('D-pad inference:')
run_inference_demo(dpad_graft, GridWorldEnvironment(grid_size=6), n_steps=10)

print('Button inference (should press ~half the time):')
run_inference_demo(button_graft, ButtonPressEnvironment(), n_steps=4)

print('MCQ inference:')
run_inference_demo(mcq_graft, MCQButtonEnvironment(), n_steps=3)

## 8 · Save & Share

Checkpoints are tiny: only the appendage MLP weights (~100 KB–1 MB each).
The VLM backbone is loaded separately from HuggingFace Hub.

In [ ]:
import os, json

save_dir = 'vla_hands_model_ckpts'
os.makedirs(save_dir, exist_ok=True)

for name, graft in [
    ('joystick', joystick_graft),
    ('dpad',     dpad_graft),
    ('button',   button_graft),
    ('mcq',      mcq_graft),
]:
    path = f'{save_dir}/{name}'
    graft.save(path)
    # Inspect saved files
    files = os.listdir(path)
    sizes = {f: os.path.getsize(f'{path}/{f}') for f in files}
    print(f'{name}: {files} — {sum(sizes.values())/1024:.1f} KB total')

print('\nCheckpoint structure for joystick:')
with open(f'{save_dir}/joystick/graft_config.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# Reload a saved appendage (demonstrates the save/load round-trip)
reloaded_graft = VLAGraft(
    vlm=vlm,
    appendage=JoystickAppendage(hidden_dim),
    config=cfg,
)
reloaded_graft.load_appendage(f'{save_dir}/joystick')
print('Reload successful!')

# Quick sanity check: forward pass using the same _preprocess helper
from vla_hands.training.trainer import _preprocess

test_env = TargetNavEnvironment()
obs = test_env.reset(seed=1)
inputs = _preprocess(processor, obs, test_env.prompt, device)
with torch.no_grad():
    action = reloaded_graft.predict_action(**inputs)
decoded = reloaded_graft.appendage.decode(action)
print(f'Test action: {decoded}')

In [ ]:
# ── Upload to HuggingFace Hub (optional) ─────────────────────────────────────
# pip install -q huggingface_hub
# from huggingface_hub import HfApi
#
# api = HfApi()
# api.upload_folder(
#     folder_path=save_dir,
#     repo_id='your-username/vla-hands-checkpoints',
#     repo_type='model',
# )
print('Uncomment the block above to push checkpoints to HuggingFace Hub.')

## 9 · Advanced: Train the Harder Environments

In [ ]:
# Spaceship has momentum — harder for BC but more realistic
# Needs more steps than basic target nav
ship_graft = VLAGraft(vlm=vlm, appendage=JoystickAppendage(hidden_dim), config=cfg)
ship_env   = SpaceshipNavEnvironment(width=224, height=224, max_steps=150)

ship_metrics = quick_train(ship_graft, ship_env, bc_steps=400, rl_steps=150,
                           tag='joystick_spaceship')

suite = BenchmarkSuite(ship_graft, processor, [ship_env], device=device)
r = suite.run_benchmark(ship_env, n_episodes=15)
print(r)

In [ ]:
# Maze needs backtracking — much harder than grid world
maze_graft = VLAGraft(vlm=vlm, appendage=DPadAppendage(hidden_dim), config=cfg)
maze_env   = MazeEnvironment(rows=5, cols=5)   # start small

maze_metrics = quick_train(maze_graft, maze_env, bc_steps=500, rl_steps=200,
                           tag='dpad_maze')

suite = BenchmarkSuite(maze_graft, processor, [maze_env], device=device)
r = suite.run_benchmark(maze_env, n_episodes=15)
print(r)

In [ ]:
# Full curriculum: gradually unfreeze VLM layers as training progresses
# Much better final performance, but slower
from vla_hands import DEFAULT_CURRICULUM

full_joystick_graft = VLAGraft(vlm=vlm, appendage=JoystickAppendage(hidden_dim), config=cfg)
config = CurriculumConfig(
    bc_steps=1500,
    rl_steps=500,
    device=device,
    freezing_stages=DEFAULT_CURRICULUM,   # gradually unfreeze up to last 6 layers
    save_dir='model_checkpoints/joystick_full_curriculum',
    eval_every=300,
    log_every=100,
)
curriculum = TrainingCurriculum(full_joystick_graft, processor, TargetNavEnvironment(), config)
# curriculum.run()   # uncomment for full training (takes ~30 min on T4)
print('Uncomment curriculum.run() for full training with layer unfreezing.')

## 9b · Touchscreen: Pointing & Tapping

Train the `TouchscreenAppendage` to tap the correct colored circle out of
several distractors.  The visual skip connection gives the head direct access
to spatial patch features — useful for precise coordinate prediction.

In [ ]:
# Visualise the pointing environment and expert policy
from vla_hands import PointingEnvironment

pointing_env = PointingEnvironment(
    width=256, height=256,
    n_distractors=3,
    circle_radius=22,
    n_colors=5,
)

# Show a few episodes and their expert tap positions
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, ax in enumerate(axes):
    obs = pointing_env.reset(seed=i * 13)
    exp_x, exp_y = pointing_env.expert_action()
    ax.imshow(obs)
    # Overlay the expert tap position
    tap_px = exp_x * pointing_env.width
    tap_py = exp_y * pointing_env.height
    ax.plot(tap_px, tap_py, 'y*', markersize=20, markeredgecolor='black')
    ax.set_title(f'seed={i*13}\n{pointing_env.prompt[:40]}...', fontsize=8)
    ax.axis('off')

plt.suptitle('PointingEnvironment — yellow star = expert tap', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Train the touchscreen graft on pointing
# The vision skip connection makes this faster to learn than LLM-only
print('='*60)
print('Training: TouchscreenAppendage → PointingEnvironment')
print('  (vision skip connection enabled)')
print('='*60)

pointing_train_env = PointingEnvironment(width=256, height=256, n_distractors=2, n_colors=4)

ts_metrics = quick_train(
    ts_graft,
    pointing_train_env,
    bc_steps=400,
    rl_steps=0,       # BC alone is often sufficient for pointing
    tag='touchscreen_pointing',
)

# Quick benchmark
from vla_hands import BenchmarkSuite
suite = BenchmarkSuite(ts_graft, processor, [pointing_train_env], device=device)
result = suite.run_benchmark(pointing_train_env, n_episodes=20)
print(f'\nPointing benchmark: success={result.success_rate:.0%}  reward={result.mean_reward:+.1f}')

In [ ]:
# Visualise touchscreen inference: predicted tap vs. target
from vla_hands.training.trainer import _preprocess

ts_graft.eval()
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    obs = pointing_train_env.reset(seed=i * 7)
    exp_x, exp_y = pointing_train_env.expert_action()

    with torch.no_grad():
        inputs = _preprocess(processor, obs, pointing_train_env.prompt, device)
        out = ts_graft(**inputs)
        pred = out['action'].squeeze(0).cpu()
        pred_x, pred_y = float(pred[0]), float(pred[1])

    w, h = pointing_train_env.width, pointing_train_env.height
    ax.imshow(obs)
    ax.plot(exp_x * w, exp_y * h, 'g*', markersize=14, label='expert', markeredgecolor='black')
    ax.plot(pred_x * w, pred_y * h, 'rx', markersize=14, label='predicted', markeredgewidth=2)

    dist = ((exp_x - pred_x)**2 + (exp_y - pred_y)**2) ** 0.5
    hit = dist * max(w, h) <= pointing_train_env.circle_radius * 1.2
    ax.set_title(f'{"✓" if hit else "✗"} dist={dist:.2f}', fontsize=10)
    ax.axis('off')

axes.flat[0].legend(loc='upper right', fontsize=8)
plt.suptitle('Touchscreen Inference: green★=expert, red✗=predicted', fontsize=12)
plt.tight_layout()
plt.show()

## 10 · What's Next?

### Features added in this notebook
| Feature | Module | Status |
|---------|--------|--------|
| 🕹️ Joystick (directional) | `JoystickAppendage` | ✅ |
| 🎮 D-pad (discrete 5-way) | `DPadAppendage` | ✅ |
| 🔘 Button (binary) | `ButtonAppendage` | ✅ |
| 🎛️ MultiButton (N independent) | `MultiButtonAppendage` | ✅ |
| 👆 **Touchscreen (absolute tap)** | `TouchscreenAppendage` | ✅ new |
| 🔁 **LoRA training** | `apply_lora()` | ✅ new |
| 🗣️ **Dynamic prompt vocab** | `PromptVocab` | ✅ new |

### Ideas to explore

```
Appendages:    Slider (1D continuous), Gyro (3-axis SO3), Pointer+Click (x,y + button)
Environments:  Snake game, berry collection (nav+button), camera pan/tilt
Training:      Multi-task — one graft on multiple environments simultaneously
               Reward shaping — use VLM's own language confidence as reward
               Self-play — environments that adapt to graft performance
Architecture:  CompositeGraft — joystick + button fires simultaneously
               Shared backbone across multiple appendages (multi-head)
               Cross-attention instead of mean-pool for vision skip
Deployment:    ONNX export, TorchScript, robot arm integration via ROS
```

**Contribute** at the GitHub repo. PRs welcome for new appendages, environments, and training recipes!

---
_vla-hands — MIT License_

## 10b · Export GIFs

In [ ]:
import os
from vla_hands.utils.gif import record_expert_gif, save_rollout_gif
from IPython.display import Image as IPImage

os.makedirs('gifs', exist_ok=True)

# Expert rollout GIFs (no VLM needed)
for tag, env in [
    ('targetnav', TargetNavEnvironment(width=200, height=200)),
    ('gridworld', GridWorldEnvironment(grid_size=6)),
]:
    record_expert_gif(env, path=f'gifs/expert_{tag}.gif', n_steps=16, seed=7, fps=6)
    print(f'Expert {tag} GIF saved.')

# Trained graft GIFs
for tag, graft, env in [
    ('joystick',  joystick_graft, TargetNavEnvironment(width=200, height=200)),
    ('dpad',      dpad_graft,     GridWorldEnvironment(grid_size=6)),
    ('touchscreen', ts_graft,     PointingEnvironment(width=200, height=200, n_distractors=3)),
]:
    save_rollout_gif(graft, processor, env,
                     path=f'gifs/trained_{tag}.gif',
                     n_steps=16, seed=42, device=device, fps=5)
    print(f'Trained {tag} GIF saved.')

print('\nDisplaying joystick expert vs trained:')
display(IPImage(filename='gifs/expert_targetnav.gif'))
display(IPImage(filename='gifs/trained_joystick.gif'))